#### Investigating the outliers in rf1, rf2, and sf2 datasets

In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.real_datamodule import RealDataModule
import numpy as np
import matplotlib.pyplot as plt
from moc.metrics.distribution_metrics import pce, multivariate_energy_score, mse
import torch

In [31]:
import wandb
wandb.login(key="4c4c0af24adae54125eb331b61a113a7fbf644d1") #NAOMI'S KEY

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/naomi/.netrc


True

In [32]:
config = get_config()
config.device = 'cuda'
data_group, data_name = 'mulan', 'rf1'
hparams = {
    'model': 'mixture',
    'prerank': 'none',
    'lambda': 0.0,
}

In [33]:
rc = RunConfig(config, data_group, data_name, hparams = hparams)
datamodule = RealDataModule(rc, num_workers=8, seed=0)
p, q = datamodule.input_dim, datamodule.output_dim #268,16

Removing 2 known outliers by content from /ssd/bdml2/naomi/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/data/outliers_x_rf1.npy
HEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEERRRRRRRRRRRREEEEEEEEEEEEEE
[array([[204.  ,  78.9 , 199.  , 109.  , 103.  ,  15.4 ,  65.4 ,  81.5 ,
        206.  ,   3.54, 191.  ,  82.4 , 103.  ,  15.6 ,  64.2 ,  80.1 ,
        208.  ,   3.59, 188.  ,  71.8 , 104.  ,  15.4 ,  64.2 ,  79.9 ,
        211.  ,   3.65, 189.  ,  69.8 , 104.  ,  15.5 ,  64.2 ,  79.6 ,
        214.  ,   3.58, 189.  ,  68.8 ,  98.1 ,  15.4 ,  64.2 ,  78.7 ,
        219.  ,   3.71, 192.  ,  68.8 , 101.  ,  15.8 ,  53.5 ,  76.4 ,
        227.  ,   3.66, 194.  ,  72.8 ,  92.3 ,  16.8 ,  53.5 ,  76.2 ,
        237.  ,   3.48, 198.  ,  78.7 ,  89.6 ,  17.1 ,  53.5 ,  74.4 ]],
      dtype=float32), array([[239.  ,   3.52, 284.  , 175.  , 112.  ,  15.5 ,  67.2 ,  82.2 ,
        220.  ,   3.5 , 264.  , 166.  , 112.  ,  15.6 ,  65.4 ,  81.9 ,
        209.  ,   3.52, 240.  , 156.  , 11

In [ ]:
model = MixtureLightningModule(p, q)
trainer = get_lightning_trainer(rc)
trainer.fit(model, datamodule)
trainer.test(model=model, datamodule=datamodule)
wandb.finish()

/home/naomi/miniconda/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/naomi/miniconda/lib/python3.12/site-packages/i ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Checking False, False


In [ ]:
ckpt_path = trainer.checkpoint_callback.best_model_path
best_model = MixtureLightningModule.load_from_checkpoint(ckpt_path)
best_model.eval().to(config.device)
test_loader = datamodule.test_dataloader()

for x, y, idx in test_loader:
    x = x.to(config.device)
    y = y.to(config.device)
    dist = best_model.predict(x)
    nll_value = -dist.log_prob(y)
    print(nll_value.mean().item())

-7.573997497558594
-7.446508407592773
-8.007024765014648
3290.384765625
-7.609258651733398
-6.98754358291626
-4.580092430114746
-7.5220770835876465
